In [ ]:
# Train the waste sorting classifier on a Kaggle GPU (version 4: TrashNet + RealWaste).
#
# Runs the project's own training script, unchanged, from a pinned tag of the public
# repository: every architecture and seed listed under `selection` in configs/config.yaml
# (ResNet18 and EfficientNet-B0, three seeds each). About 1-1.5 hours on a T4.
#
# Before running: in the right-hand panel, set Accelerator -> GPU (T4) and Internet -> On.
# After running: download run.zip from the Output panel (/kaggle/working/run.zip).
# (The project then unzips it into artifacts/kaggle-v4/ and runs `make select export`.)
print("ok")

In [ ]:
# 1. A GPU must be available; otherwise stop here rather than train for hours on a CPU.
import sys

import torch

assert torch.cuda.is_available(), "No GPU: set Accelerator -> GPU in the right-hand panel"
print("Python", sys.version.split()[0], "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. The code, from GitHub, at a fixed tag: rerunning this notebook trains the same code.
#    Cloned to /tmp so the dataset is not saved as notebook output.
REPO = "https://github.com/vladzpop11/waste-sorting-classifier.git"
TAG = "train-v4"
%cd /kaggle/working
# (Leave /tmp/repo before replacing it, so that running this cell again works.)
!rm -rf /tmp/repo && git clone -q --depth 1 --branch {TAG} {REPO} /tmp/repo
%cd /tmp/repo
!git log -1 --format='commit %H (%s)'

In [ ]:
# 3. Kaggle already has PyTorch, torchvision, scikit-learn, pandas, Pillow and PyYAML.
#    The package itself is not installed: Kaggle runs Python 3.13, which the Intel-Mac pins
#    in pyproject.toml exclude, so the code runs from source (PYTHONPATH=src) instead.
#    Versions as in uv.lock.
!pip install -q "mlflow-skinny==3.17.1" "sqlalchemy==2.1.5" "alembic==1.20.0"

In [ ]:
# 4. Download TrashNet and RealWaste (689 MB) and check them: archive SHA-256s, published
#    class counts; then the training script checks every image against the committed split.
!PYTHONPATH=src python -m waste_sorting_classifier.data

In [ ]:
# 5. Train every architecture with every seed listed in the config (early stopping on
#    validation loss). The choice between them is made later, on validation only, by
#    `make select` with the rule recorded in configs/config.yaml.
import os
import subprocess

import yaml

with open("configs/config.yaml") as f:
    selection = yaml.safe_load(f)["selection"]
for arch in selection["archs"]:
    for seed in selection["seeds"]:
        out = f"/kaggle/working/run/{arch}/seed{seed}"
        cmd = [sys.executable, "-m", "waste_sorting_classifier.train", "--arch", arch,
               "--seed", str(seed), "--output-dir", out]  # fmt: skip
        env = {**os.environ, "PYTHONPATH": "src"}
        pipe = {"stdout": subprocess.PIPE, "stderr": subprocess.STDOUT}
        with subprocess.Popen(cmd, env=env, text=True, **pipe) as proc:
            for line in proc.stdout:
                print(line, end="")
        assert proc.returncode == 0, f"training {arch} seed {seed} failed"

In [ ]:
# 6. One file to download: run.zip (per architecture: model.pt, metrics, history,
#    confusion matrix, MLflow store).
!git -C /tmp/repo rev-parse HEAD > /kaggle/working/run/commit.txt
!cd /kaggle/working && rm -f run.zip && zip -qr run.zip run
!ls -laR /kaggle/working/run | head -40 && ls -la /kaggle/working/run.zip